# Strategic label lab — VLM + SAM3 + ego → `g_str` / `a_str`

**Purpose (PI):** *"review if the pipeline is extracting the right
information and optimizing it until the correct strategic vocabulary is
generated in the right format, then we can scale."*

Per clip, three legs run **sequentially** (the 16 GB T4 fits them one at a
time, never together), then fuse:

| leg | engine | provenance tag |
|---|---|---|
| VLM | Qwen3.5-9B, **unsloth 4-bit** (~7–8 GB), grammar-constrained B1–B4 calls from `ph0_v2.py` | `vlm` |
| SAM3 | `facebook/sam3` text-prompted concepts (~3 GB), `ph0_sam3.py` | `sam3` |
| ego | engine-A geometry + spine (`ph0_pilot.py`, `ph1_fuse.py`), yaw vote with `strategic_gt.py` thresholds | `ego` (privileged: labels-only) |

Fusion is `ph1_fuse.py`'s own 2-of-3 voting; the S2 tokens come ONLY from
`colab/s2_schema.py` (**PROVISIONAL** — swaps in one file when the S2-gap
agent's `S2_STRATEGIC_GAP.md` lands). Every token carries **per-token
provenance** for the S-S gate's goal-provenance audit. Labels may use ego;
the goal fields never carry situation-classifier output (asserted per
record).

The **review sheet** at the end renders frames + tokens + provenance +
corroborations per clip — that is the artifact to judge the pipeline by.
Banked per clip to `Sayood/tanitad-s2-lab/` with a run manifest; restart =
re-run all cells (far-side resume).

**Prompt iteration:** the B1–B4 prompts live in `stack/scripts/ph0_v2.py`
(`P_B1…P_B4`) on this same Drive — edit there (from any box), wait for Drive
sync, then `import importlib, ph0_v2; importlib.reload(ph0_v2)` and re-run
the VLM-leg cell. That is the optimise loop.

In [ ]:
# --- parameters -------------------------------------------------------------
import os
SMOKE = os.environ.get('S2_SMOKE', '0') == '1'
N = int(os.environ.get('S2_N', '1' if SMOKE else '4'))   # clips this run
CLIP_IDS = [c for c in os.environ.get('S2_CLIPS', '').split(',') if c]
ALLOW_FALLBACK = os.environ.get('S2_ALLOW_FALLBACK', '0') == '1'
EGO_IN_PROMPT = 'past'      # ph0_v2 production setting (speed-redacted to B2)
print(f'SMOKE={SMOKE} N={N} CLIP_IDS={CLIP_IDS or "(auto: sam3-covered)"} '
      f'ALLOW_FALLBACK={ALLOW_FALLBACK}')

In [ ]:
# --- Drive mount + repo imports ---------------------------------------------
import json, sys, time
from pathlib import Path
try:
    import s2_lab_lib as L
except ImportError:
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/SayBouBase/raw/Projects/'
                       'TanitAD/colab')
    import s2_lab_lib as L
ROOT = L.add_stack_paths()
print('repo root:', ROOT)
L.pip_install_colab(SMOKE)
import s2_schema
print('schema:', s2_schema.SCHEMA_VERSION,
      '| v6 drift:', s2_schema.check_v6_drift())

In [ ]:
# --- auth + bank target -----------------------------------------------------
api = L.hf_api()
WORK = Path('/content/lab') if L.in_colab() else \
    ROOT / 'colab' / '_smoke_work' / 'lab'
WORK.mkdir(parents=True, exist_ok=True)
BANK_REPO = L.DS_LAB
BANK_PREFIX = (L.SMOKE_PREFIX + 'lab/') if SMOKE else L.LAB_PREFIX
L.ensure_repo(api, BANK_REPO)
print(f'banking to {BANK_REPO}/{BANK_PREFIX}')

In [ ]:
# --- clip selection + resume ------------------------------------------------
# Default pool: the SAM3-COVERED fused clips, so all three legs have ground
# to compare against. Override with S2_CLIPS=<id,id,...>.
ls = json.load(open(L.hf_download(L.DS_LABELS,
                                  L.FUSED_PREFIX + '_label_sources.json')))
covered = sorted(c for c, s in ls['sources'].items() if s.get('sam3'))
pool = CLIP_IDS or covered
done = L.done_set(api, BANK_REPO, BANK_PREFIX, suffix='.s2.json')
clips = [c for c in pool if c not in done][:N]
print(f'pool {len(pool)} · far-side done {len(done)} · this run {len(clips)}')
for c in clips:
    print('  ', c)

In [ ]:
# --- inputs: ego npz (always) + video frames + Alpamayo ---------------------
inputs, alp_by = {}, {}
if clips and SMOKE:
    alp_by = {c: L.stub_alpamayo(c) for c in clips}
elif clips:
    REC_PQ = str(WORK / 'records.parquet')
    if not Path(REC_PQ).exists():
        import shutil
        shutil.copyfile(L.hf_download(L.DS_ALP, 'records.parquet'), REC_PQ)
    alp_all = L.load_alpamayo(REC_PQ)
    alp_by = {c: alp_all.get(c) for c in clips}
    loc = L.w120_locations(api)
for cid in clips:
    ego_p = L.hf_download(L.DS_LABELS, f'{L.EGO_PREFIX}{cid}.npz')
    if SMOKE:
        frames, n_past = L.stub_frames(), 16
    else:
        cw = WORK / cid[:8]
        L.bridge_batch([cid], loc, REC_PQ, cw)
        import ph0_pilot
        frames, _t, n_past = ph0_pilot.sample_clip_frames(
            str(cw / 'videos' / f'{cid}.mp4'), t0_s=8.0)
    inputs[cid] = {'ego_npz': ego_p, 'frames': frames, 'n_past': n_past}
print(f'inputs ready for {len(inputs)} clips '
      f'(alpamayo present: {sum(1 for c in inputs if alp_by.get(c))})')

In [ ]:
# --- leg 1 (0-GPU): ego geometry — engine A + spine + yaw vote --------------
ego_by = {c: L.ego_leg(inputs[c]['ego_npz']) for c in inputs}
for c, e in ego_by.items():
    v = e['g_str_vote']
    print(f'{c[:8]} ego vote {v["token"]} {v.get("args") or ""} '
          f'(net dyaw {v["net_dyaw_deg_from_t0"]}°) · spine '
          + json.dumps((e['spine'] or {}).get('speed_profile', {}))[:110])

In [ ]:
# --- leg 2: VLM (unsloth 4-bit Qwen3.5-9B), grammar-constrained B1–B4 -------
# Resolution is at RUNTIME from a candidate list, newest first; what loaded
# is PRINTED; an older-generation substitute needs ALLOW_FALLBACK=True.
resolved = L.resolve_vlm_model(api, allow_fallback=ALLOW_FALLBACK)
if SMOKE:
    print('[vlm] SMOKE — resolver ran for real (above); records are stubs')
    v2_by = {c: L.stub_vlm_record(c) for c in inputs}
else:
    vlm = L.load_vlm(resolved['model_id'])
    L.gpu_mem_report('vlm load')
    v2_by = {}
    for cid in inputs:
        rec = L.vlm_leg(vlm, inputs[cid]['frames'], inputs[cid]['n_past'],
                        ego_by[cid]['engine_a'], ego_by[cid]['ego_state'],
                        EGO_IN_PROMPT)
        rec['clip_id'] = cid
        v2_by[cid] = rec
        print(f'[vlm] {cid[:8]} all_valid={rec.get("_all_valid")} '
              f'goal={((rec.get("symbols") or {}).get("goal_kind"))}')
    L.gpu_mem_report('vlm leg total')
    L.free_leg(vlm)                    # sequential legs: free before SAM3
    L.gpu_mem_report('after vlm free')

In [ ]:
# --- leg 3: SAM3 (text-prompted concepts + B3 box cross-check) --------------
if SMOKE:
    sam3_by = {c: L.stub_sam3_record(c) for c in inputs}
else:
    proc, _m = L.load_sam3()
    L.gpu_mem_report('sam3 load')
    sam3_by = {}
    for cid in inputs:
        sam3_by[cid] = L.sam3_leg(proc, inputs[cid]['frames'], v2_by[cid])
        h = sam3_by[cid]['per_concept_hits']
        print(f'[sam3] {cid[:8]} ' + (', '.join(
            f'{k}:{v}' for k, v in h.items() if v) or
            'no detections (valid abstention)'))
    L.gpu_mem_report('sam3 leg total')
    L.free_leg(proc)
    L.gpu_mem_report('after sam3 free')

In [ ]:
# --- fuse (ph1_fuse 2-of-3 voting) -> S2 record -> BANK PER CLIP ------------
entries = []
for cid in inputs:
    fused = L.fuse_one(v2_by[cid], sam3_by.get(cid), inputs[cid]['ego_npz'],
                       alp_by.get(cid))
    s2 = L.to_s2(fused, ego_extra=ego_by[cid])
    sz = L.bank_json(api, BANK_REPO, f'{BANK_PREFIX}{cid}.s2.json',
                     {**s2, '_fused': fused, '_smoke': SMOKE})
    entries.append({'s2': s2, 'fused': fused,
                    'frames': inputs[cid]['frames'], 'smoke': SMOKE})
    g = s2['g_str']
    print(f'[bank] {cid[:8]} {sz} B far-side-verified')
    print(f'   g_str {g["token"]} {g["args"] or ""} prov={g["provenance"]} '
          f'conf={g["confidence"]}')
    print('   a_str', [(a['token'], a['args'], a['provenance'])
                       for a in s2['a_str']] or '(none)')

In [ ]:
# --- the REVIEW SHEET: frames + tokens + provenance + corroborations --------
html = L.review_sheet_html(
    entries, title=f'S2 label lab — {time.strftime("%Y-%m-%d %H:%M")} '
                   f'({"SMOKE" if SMOKE else "T4"})')
sheet_path = str(WORK / 'review_sheet.html')
L.show_html(html, sheet_path)          # renders inline in Colab + saves
api.upload_file(path_or_fileobj=sheet_path,
                path_in_repo=f'{BANK_PREFIX}_sheets/'
                             f'{time.strftime("%Y%m%d-%H%M%S")}.html',
                repo_id=BANK_REPO, repo_type='dataset')
print('sheet banked beside the records')

In [ ]:
# --- run manifest + resume proof --------------------------------------------
man_rf = L.run_manifest(api, BANK_REPO, BANK_PREFIX, 'label-lab', {
    'smoke': SMOKE, 'n_clips': len(entries),
    'clips': [e['s2']['clip_id'] for e in entries],
    'vlm_resolved': resolved, 'schema_version': s2_schema.SCHEMA_VERSION,
    'ego_in_prompt': EGO_IN_PROMPT,
    'evidence_class': 'SMOKE-STUB' if SMOKE else 'MEASURED'})
done2 = L.done_set(api, BANK_REPO, BANK_PREFIX, suffix='.s2.json')
for e in entries:
    assert e['s2']['clip_id'] in done2, \
        f'banked clip {e["s2"]["clip_id"]} MISSING from far-side done-set!'
print(f'resume check: far side holds {len(done2)} records; this run\'s '
      'clips all present -> a session death now costs nothing')
print(f'manifest: {man_rf}')
print('LAB_DONE')